# Stage 4 — Resilience Analysis + All-Zone Coverage Report

**Project:** Dark Store Placement + Integrated Forward & Reverse Logistics  
**Author:** Sneha  

**Depends on:**
- `data/master_df_v2.parquet` — with `dark_store_id` column 
- `data/dark_stores_final.csv` — finalised dark store locations

**Outputs:**
- `outputs/resilience_analysis.png` — failure simulation + K±1 robustness chart

---

**What this notebook does (plain English):**
1. **Dark store failure simulation** — removes one dark store at a time, redistributes customers to the remaining K-1 stores, measures how much coverage drops and how far customers now have to travel
2. **K±1 robustness test** — shows that our chosen K is better than K-1 and K+1, proving the choice is optimal
3. Saves `resilience_analysis.png` with both panels

---
## 0. Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.clustering import (
    load_sp_data,
    build_zip_level_coords,
    run_kmeans,
    assign_voronoi,
    compute_coverage,
    haversine_km,
)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

DATA_DIR   = '../data'
OUTPUT_DIR = '../outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Imports OK.')

---
## 1. Load data

In [ ]:
df = pd.read_parquet(f'{DATA_DIR}/master_df_v2.parquet')
dark_stores_df = pd.read_csv(f'{DATA_DIR}/dark_stores_final.csv')

centroids = dark_stores_df[['lat', 'lon']].values
K = len(centroids)
customer_coords = df[['customer_lat', 'customer_lon']].values

print(f'Loaded {len(df):,} customer rows')
print(f'K = {K} dark stores')
print()
print('Dark stores:')
print(dark_stores_df[['dark_store_id','lat','lon','coverage_5km_pct']].to_string(index=False))

---
## 2. Failure simulation

**What we do:** Remove one dark store at a time (simulate it going offline — power cut, fire, flood).  
Redistribute all its customers to the nearest remaining K-1 stores.  
Measure: how much does coverage drop? How much does the average travel distance increase?

In [ ]:
# Baseline metrics (all K stores operating)
baseline_coverage = compute_coverage(customer_coords, centroids, radius_km=5.0)
baseline_zone_ids = assign_voronoi(customer_coords, centroids)
baseline_dists = haversine_km(
    customer_coords[:, 0], customer_coords[:, 1],
    centroids[baseline_zone_ids, 0], centroids[baseline_zone_ids, 1],
)
baseline_mean_dist = float(baseline_dists.mean())

print(f'Baseline (all {K} stores operating):')
print(f'  Coverage (5km)    : {baseline_coverage*100:.1f}%')
print(f'  Mean distance     : {baseline_mean_dist:.3f} km')
print()

# Simulate failure of each dark store
failure_results = []

for failed_store in range(K):
    # Keep all centroids except the failed one
    surviving_centroids = np.delete(centroids, failed_store, axis=0)

    # Reassign all customers to nearest surviving store
    new_zone_ids = assign_voronoi(customer_coords, surviving_centroids)

    # Compute coverage and mean distance under failure
    new_coverage = compute_coverage(customer_coords, surviving_centroids, radius_km=5.0)
    new_dists = haversine_km(
        customer_coords[:, 0], customer_coords[:, 1],
        surviving_centroids[new_zone_ids, 0], surviving_centroids[new_zone_ids, 1],
    )
    new_mean_dist = float(new_dists.mean())

    coverage_drop = (baseline_coverage - new_coverage) * 100
    dist_increase_pct = (new_mean_dist - baseline_mean_dist) / baseline_mean_dist * 100

    failure_results.append({
        'failed_store':       f'DS-{failed_store}',
        'coverage_pct':       round(new_coverage * 100, 1),
        'coverage_drop_pct':  round(coverage_drop, 1),
        'mean_dist_km':       round(new_mean_dist, 3),
        'dist_increase_pct':  round(dist_increase_pct, 1),
    })
    print(f'  DS-{failed_store} fails → coverage={new_coverage*100:.1f}% '
          f'(drop={coverage_drop:.1f}%) | '
          f'mean dist={new_mean_dist:.3f}km (+{dist_increase_pct:.1f}%)')

failure_df = pd.DataFrame(failure_results)
print()
print('Summary:')
print(failure_df.to_string(index=False))

---
## 3. K±1 robustness test

**What we do:** Run K-Means with K-1 and K+1 stores.  
Show that our chosen K gives better coverage than K-1, and is not significantly worse than K+1.  
This proves our K choice is robust.

In [ ]:
df_sp = load_sp_data(f'{DATA_DIR}/master_df_v2.parquet')
zip_df = build_zip_level_coords(df_sp)
coords_zip = zip_df[['lat', 'lon']].values
weights_zip = zip_df['demand_weight'].values

# Test K-1, K, K+1
k_test_range = range(max(2, K - 1), K + 2)
print(f'Testing K = {list(k_test_range)} (our chosen K = {K})')
kmeans_kpm1 = run_kmeans(coords_zip, weights_zip, k_range=k_test_range)

kpm1_results = []
for k, res in kmeans_kpm1.items():
    cov = compute_coverage(customer_coords, res['centroids'], radius_km=5.0)
    zone_ids = assign_voronoi(customer_coords, res['centroids'])
    dists = haversine_km(
        customer_coords[:, 0], customer_coords[:, 1],
        res['centroids'][zone_ids, 0], res['centroids'][zone_ids, 1],
    )
    label = f'K={k} (optimal)' if k == K else f'K={k}'
    kpm1_results.append({
        'k':              k,
        'label':          label,
        'coverage_pct':   round(cov * 100, 1),
        'mean_dist_km':   round(float(dists.mean()), 3),
        'silhouette':     round(res['silhouette'], 4),
    })
    print(f'  K={k}: coverage={cov*100:.1f}% | mean_dist={dists.mean():.3f}km | silhouette={res["silhouette"]:.4f}')

kpm1_df = pd.DataFrame(kpm1_results)
print()
print(kpm1_df.to_string(index=False))

---
## 4. Save resilience_analysis.png

In [ ]:
ZONE_COLOURS = [
    '#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#264653',
    '#F4A261', '#A8DADC', '#6D6875', '#B5838D', '#E07A5F',
    '#8338EC', '#FB5607',
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel 1: Coverage drop when each store fails ─────────────────────────────
ax = axes[0]
bar_colours = [ZONE_COLOURS[i % len(ZONE_COLOURS)] for i in range(K)]
bars = ax.bar(
    failure_df['failed_store'],
    failure_df['coverage_drop_pct'],
    color=bar_colours, edgecolor='white', linewidth=0.8
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Failed dark store', fontsize=12)
ax.set_ylabel('Coverage drop (percentage points)', fontsize=12)
ax.set_title(
    'Coverage degradation\nwhen one store fails',
    fontsize=12, fontweight='bold'
)
# Annotate bars
for bar, val in zip(bars, failure_df['coverage_drop_pct']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'-{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

# ── Panel 2: Mean distance increase when each store fails ────────────────────
ax2 = axes[1]
bars2 = ax2.bar(
    failure_df['failed_store'],
    failure_df['dist_increase_pct'],
    color=bar_colours, edgecolor='white', linewidth=0.8
)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Failed dark store', fontsize=12)
ax2.set_ylabel('Mean distance increase (%)', fontsize=12)
ax2.set_title(
    'Mean travel distance increase\nwhen one store fails',
    fontsize=12, fontweight='bold'
)
for bar, val in zip(bars2, failure_df['dist_increase_pct']):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'+{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

# ── Panel 3: K±1 robustness — coverage by K ──────────────────────────────────
ax3 = axes[2]
k_vals = kpm1_df['k'].tolist()
cov_vals = kpm1_df['coverage_pct'].tolist()
panel3_colours = ['crimson' if k == K else 'steelblue' for k in k_vals]
bars3 = ax3.bar(
    [str(k) for k in k_vals],
    cov_vals,
    color=panel3_colours, edgecolor='white', linewidth=0.8
)
ax3.axhline(70, color='green', linestyle='--', linewidth=1.5, label='70% target')
ax3.set_xlabel('Number of dark stores (K)', fontsize=12)
ax3.set_ylabel('5-km coverage (%)', fontsize=12)
ax3.set_title(
    f'K±1 robustness — optimal K={K} highlighted',
    fontsize=12, fontweight='bold'
)
ax3.legend(fontsize=10)
ax3.bar([], [], color='crimson', label=f'Chosen K={K}')
ax3.bar([], [], color='steelblue', label='Other K values')
ax3.legend(fontsize=10)
for bar, val in zip(bars3, cov_vals):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.suptitle(
    f'Resilience Analysis — Dark Store Network (K={K})',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
save_path = f'{OUTPUT_DIR}/resilience_analysis.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/resilience_analysis.png')

---
## 5. Day 4 Deliverables Checklist

In [ ]:
files_to_check = [
    (f'{OUTPUT_DIR}/resilience_analysis.png', 'resilience_analysis.png'),
    (f'{DATA_DIR}/dark_stores_final.csv',     'dark_stores_final.csv (from Day 3)'),
    (f'{DATA_DIR}/master_df_v2.parquet',      'master_df_v2.parquet (from Day 3)'),
]

print('Day 4 Deliverables Checklist')
print('=' * 50)
all_ok = True
for path, label in files_to_check:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {label}')
    if not exists:
        all_ok = False

print()
print('Key resilience findings:')
worst_failure = failure_df.loc[failure_df['coverage_drop_pct'].idxmax()]
print(f'  Worst single-store failure : {worst_failure["failed_store"]} '
      f'(coverage drops by {worst_failure["coverage_drop_pct"]}%)')
print(f'  Max distance increase      : {failure_df["dist_increase_pct"].max():.1f}%')
k_minus1_cov = kpm1_df[kpm1_df['k'] == K-1]['coverage_pct'].values[0] if K-1 in kpm1_df['k'].values else 'N/A'
k_cov = kpm1_df[kpm1_df['k'] == K]['coverage_pct'].values[0]
print(f'  Coverage at K={K} (optimal) : {k_cov}%')
if k_minus1_cov != 'N/A':
    print(f'  Coverage at K={K-1}         : {k_minus1_cov}%  (confirming K={K} is better)')
print()
if all_ok:
    print('🎉 All Day 4 outputs complete! sensitivity_report_section.docx was generated separately.')
else:
    print('⚠  Some files are missing.')